# EasyOPD — One Entry Point for Every OPD Setting

A unified on-policy distillation framework on top of **verl**. The *same* API selects any method—only the **name + YAML** change.

This demo shows three settings and their baselines through one interface:

| Setting | Method | Baselines |
|---|---|---|
| Cross-tokenizer OPD | **SimCT** | ULD, ALM, DSKD |
| On-policy self-distillation | **SDPO** | GRPO |
| Step-wise OPD | **SOD** | response-level OPD, GRPO |

### Setup — put the repo root on the path
So `import easyopd` and the relative `easyopd/config/*.yaml` paths resolve when the notebook lives in `examples/`.

In [ ]:
# Robust paths that do NOT depend on the kernel's working directory.
# `easyopd` is already importable in this env; derive everything from its location.
import os, sys, easyopd
PKG  = os.path.dirname(easyopd.__file__)          # .../EasyOPD/easyopd
REPO = os.path.dirname(PKG)                        # .../EasyOPD
CFG  = os.path.join(PKG, 'config')                 # .../EasyOPD/easyopd/config
# bundled run logs: prefer the copy next to this notebook, else the repo-level one
LOGS = next((d for d in (
    os.path.join(REPO, 'examples', 'demo', 'demo_logs'),
    os.path.join(REPO, 'examples', 'demo_logs'),
) if os.path.isdir(d)), os.path.join(REPO, 'examples', 'demo', 'demo_logs'))
os.chdir(REPO)                                     # so `bash examples/...` in step 3 works
if REPO not in sys.path:
    sys.path.insert(0, REPO)
print('repo root:', REPO)
print('config dir:', CFG)
print('logs dir  :', LOGS)

### 0. Warm-up (run once before recording)
The first discovery imports all method modules; subsequent calls are instant.

In [ ]:
from easyopd import EasyOPD
_ = EasyOPD.list_methods()  # warm the registry cache (slow only the first time)

### 1. Discover all released methods

In [ ]:
EasyOPD.list_methods()

### 2. The *same* call loads any method — only the name + YAML change

In [ ]:
from easyopd import EasyOPD
# Absolute config paths (cwd-independent).
# (a) Cross-tokenizer OPD   (baselines: uld | alm | dskd)
simct = EasyOPD.from_hparams("simct", config_path=os.path.join(CFG, "simct.yaml"), auto_resolve_data=False)
# (b) On-policy self-distillation   (baseline: grpo)
sdpo  = EasyOPD.from_hparams("sdpo",  config_path=os.path.join(CFG, "sdpo.yaml"),  auto_resolve_data=False)
# (c) Step-wise OPD   (baselines: response-level opd, grpo)
sod   = EasyOPD.from_hparams("sod",   config_path=os.path.join(CFG, "sod.yaml"),   auto_resolve_data=False)

for m in (simct, sdpo, sod):
    print(f"{m.method_name:6s} -> loss_mode={m.config['method'].get('loss_mode', m.method_name)}")

### 3. Launch training — one script per method; the config selects the hooks

```bash
bash examples/simct/run_simct.sh   # cross-tokenizer: uld|alm|dskd
bash examples/sdpo/run_sdpo.sh     # self-distillation: grpo
bash examples/sod/run_sod.sh       # step-wise: response-level opd, grpo
```

We launch SimCT live and watch until the first training metrics appear
(model loading + rollout take a few minutes; interrupt the kernel once the
first `simct/xtok_kd_loss` line prints).

In [ ]:
# Uncomment to launch live during recording:
# !bash examples/simct/run_simct.sh

### 4. Supervision-specific diagnostics

EasyOPD activates **only the hooks each method needs** and logs metrics you cannot
read off task accuracy. Summary of real 2-step runs (`examples/demo_logs/`):

In [ ]:
diagnostics = {
    "SimCT  cross-tokenizer  (alignment + teacher sidecar + loss)": [
        "Cross-tokenizer teacher sidecar enabled for loss_mode=simct",
        "loading teacher lm_head from Qwen2.5-7B-Instruct onto cuda:0",
        "simct/xtok_kd_loss  |  simple/teacher_loss_tokens_mean: 397.75",
    ],
    "SDPO   self-distillation  (EMA self-teacher + reprompt)": [
        "building EMA self-teacher from Qwen3-8B",
        "[EMA-check] teacher=37 student=37 shared=37  (weights tied)",
        "actor/sdpo/loss: 0.0047  teacher_fraction: 0.25",
        "self_distillation/reprompt_sample_fraction: 0.125",
    ],
    "SOD    step-wise  (per-token KL reweighting)": [
        "stepwise_opd_coef = 1.0",
        "actor/token_kl/mean: -0.42  max: 14.02  min: -34.06  (step-level divergence)",
        "Training Progress: 100%  2/2 steps  (checkpoint saved)",
    ],
}
for title, lines in diagnostics.items():
    print(f"\n=== {title} ===")
    for ln in lines:
        print("  " + ln)

*(optional)* the same lines straight from the real run logs:

In [ ]:
for name in ("simct", "sdpo", "sod"):
    print(f"\n=== {name}_diagnostics.txt ===")
    print(open(os.path.join(LOGS, f"{name}_diagnostics.txt")).read().rstrip())

**Same entry point, three supervision regimes.** Switch methods by changing the YAML only.

🔗 https://github.com/lds-ustc/EasyOPD